<a href="https://colab.research.google.com/github/yooongZa/AIFFEL_Quest_EPA/blob/main/NSMC_SentencePiece_Project_fact_Konly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 네이버 영화리뷰 감정 분석 문제에 SentencePiece 적용해 보기

```text
- 네이버 영화리뷰 감정 분석 코퍼스에 SentencePiece를 적용시킨 모델 학습하기
- 학습된 모델로 sp_tokenize() 메소드 구현하기
- 구현된 토크나이저를 적용하여 네이버 영화리뷰 감정 분석 모델을 재학습하기
- KoNLPy 형태소 분석기를 사용한 모델과 성능 비교하기
- SentencePiece 모델의 model_type, vocab_size 등을 변경해 가면서 성능 개선 여부 확인하기  
```
```text
- SentencePiece를 이용하여 모델을 만들기까지의 과정이 정상적으로 진행되었는가?  
 :코퍼스 분석, 전처리, SentencePiece 적용, 토크나이저 구현 및 동작이 빠짐없이 진행되었는지를 봅니다.
- SentencePiece를 통해 만든 Tokenizer가 자연어처리 모델과 결합하여 동작하는가?  
 :SentencePiece 토크나이저가 적용된 Text Classifier 모델이 정상적으로 수렴하여 80% 이상의 test accuracy가 확인되었다면 성공입니다.
- SentencePiece의 성능을 다각도로 비교분석하였는가?  
 :SentencePiece 토크나이저를 활용했을 때의 성능을 다른 토크나이저 혹은 SentencePiece의 다른 옵션의 경우와 비교하여 분석을 체계적으로 진행하였는지 봅니다.

In [1]:
import sys

try:
    import sentencepiece
except ImportError:
    !{sys.executable} -m pip install -q sentencepiece==0.2.2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from IPython.display import display

SEED = 42
torch.manual_seed(SEED)

# Colab GPU가 있으면 CUDA, 없으면 CPU를 사용합니다.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_DIR = Path("/content/sentencepiece_basic") if Path("/content").exists() else Path("/tmp/sentencepiece_basic")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

MAX_LENGTH = 80
BATCH_SIZE = 512
EPOCHS = 3

print("device:", DEVICE)
print("work directory:", WORK_DIR)


device: cuda
work directory: /content/sentencepiece_basic


### 1. NSMC 데이터 불러오기


In [2]:
NSMC_COMMIT = "cc0670e872d4ac27bfe36c87456783004b39ef6c"
TRAIN_URL = f"https://raw.githubusercontent.com/e9t/nsmc/{NSMC_COMMIT}/ratings_train.txt"
TEST_URL = f"https://raw.githubusercontent.com/e9t/nsmc/{NSMC_COMMIT}/ratings_test.txt"

train_data = pd.read_table(TRAIN_URL)
test_data = pd.read_table(TEST_URL)

# document가 비어 있는 행과 같은 문장이 반복된 행을 제거합니다.
train_data = train_data.dropna(subset=["document", "label"])
test_data = test_data.dropna(subset=["document", "label"])
train_data = train_data.drop_duplicates(subset=["document"])
test_data = test_data.drop_duplicates(subset=["document"])

# train에 있던 문장이 test에도 있으면 test에서 제외합니다.
test_data = test_data[~test_data["document"].isin(train_data["document"])]

# train 중 5,000개를 validation으로 사용합니다.
validation_data = train_data.sample(n=5000, random_state=SEED)
train_data = train_data.drop(validation_data.index)

train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print("train:", len(train_data))
print("validation:", len(validation_data))
print("test:", len(test_data))

train: 141182
validation: 5000
test: 48361


In [3]:
display(train_data.head(3))
print("train 긍정 비율:", round(train_data["label"].mean(), 3))
print("validation 긍정 비율:", round(validation_data["label"].mean(), 3))
print("test 긍정 비율:", round(test_data["label"].mean(), 3))

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0


train 긍정 비율: 0.498
validation 긍정 비율: 0.506
test 긍정 비율: 0.503


### 2. Unigram SentencePiece 학습

- `filtered_corpus`: 앞 단계에서 정제한 문장 목록
- `temp_file`: SentencePiece가 읽을 text 파일
- `vocab_size = 8000`: 만들 token 조각 수
- `model_type`을 쓰지 않으면 LMS 설명처럼 Unigram이 기본값

In [4]:
filtered_corpus = train_data["document"].tolist()

import sentencepiece as spm
import os
temp_file = 'korean-english-park.train.ko.temp'

vocab_size = 8000

with open(temp_file, 'w', encoding='utf-8') as f:
    for row in filtered_corpus:   # 이전에 나왔던 정제했던 corpus를 활용해서 진행해야 합니다.
        f.write(str(row) + '\n')

spm.SentencePieceTrainer.Train(
    '--input={} --model_prefix=korean_spm --vocab_size={} --pad_id=3 --minloglevel=2'.format(
        temp_file, vocab_size
    )
)
# 위 Train에서 --model_type=unigram이 default(기본값)입니다.

!ls -l korean_spm*

-rw-r--r-- 1 root root 377698 Sep  3 01:12 korean_spm.model
-rw-r--r-- 1 root root 144719 Sep  3 01:12 korean_spm.vocab


### 3. Encode/Decode 확인

- `EncodeAsIds()`: 문장을 숫자 ID 목록으로 변경
- `SampleEncodeAsPieces()`: 문장을 사람이 읽을 수 있는 subword(부분단어)로 표시
- `DecodeIds()`: ID 목록을 다시 문장으로 복원

In [5]:
s = spm.SentencePieceProcessor()
s.Load('korean_spm.model')

# SentencePiece를 활용한 sentence -> encoding
tokensIDs = s.EncodeAsIds('아버지가방에들어가신다.')
print(tokensIDs)

# SentencePiece를 활용한 sentence -> encoded pieces
print(s.SampleEncodeAsPieces('아버지가방에들어가신다.', -1, 0.1))

# SentencePiece를 활용한 encoding -> sentence 복원
print(s.DecodeIds(tokensIDs))

[1406, 12, 387, 16, 1317, 12, 131, 19, 5]
['▁', '아', '버', '지', '가', '방', '에', '들어', '가', '신', '다', '.']
아버지가방에들어가신다.


### 4.`sp_tokenize()` 구현

1. `EncodeAsIds()` 결과인 Python `list`를 `torch.Tensor`로 바꿉니다.
2. padding에 `<unk>`의 ID인 `0`이 아니라 `s.pad_id()`인 `3`을 사용합니다.


- `[:MAX_LENGTH]`: 아주 긴 리뷰를 앞의 80 token까지만 사용해 메모리를 제한합니다.
- `encoding='utf-8'`: 한국어 vocab 파일을 같은 인코딩으로 읽습니다.
- `vocab_path`: 뒤에서 BPE vocabulary도 같은 함수로 읽습니다.

In [6]:
def sp_tokenize(s, corpus, vocab_path="./korean_spm.vocab"):

    tensor = []

    for sen in corpus:
        token_ids = s.EncodeAsIds(sen)[:MAX_LENGTH]
        tensor.append(torch.tensor(token_ids, dtype=torch.long))

    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = f.readlines()

    word_index = {}
    index_word = {}

    for idx, line in enumerate(vocab):
        word = line.split("\t")[0]

        word_index.update({word: idx})
        index_word.update({idx: word})

    tensor = pad_sequence(
        tensor,
        batch_first=True,
        padding_value=s.pad_id(),
    )

    return tensor, word_index, index_word

In [7]:
practice_sentences = [
    "아버지가방에들어가신다.",
    "이 영화는 정말 재미있어요.",
    "별로였어요.",
]

practice_tensor, word_index, index_word = sp_tokenize(s, practice_sentences)

print("tensor shape:", practice_tensor.shape)
print(practice_tensor)
print("vocab size:", len(word_index))

first_ids = [
    token_id
    for token_id in practice_tensor[0].tolist()
    if token_id != s.pad_id()
]
print("첫 문장 복원:", s.DecodeIds(first_ids))
assert s.DecodeIds(first_ids) == practice_sentences[0]

tensor shape: torch.Size([3, 9])
tensor([[1406,   12,  387,   16, 1317,   12,  131,   19,    5],
        [  28,  127,   29, 1488,    5,    3,    3,    3,    3],
        [ 213, 2340,    5,    3,    3,    3,    3,    3,    3]])
vocab size: 8000
첫 문장 복원: 아버지가방에들어가신다.


### 5. `Dataset`과 `DataLoader`


```text
__len__()     → 데이터가 총 몇 개인지 반환
__getitem__() → i번째 (입력, 정답) 쌍 반환
DataLoader    → 여러 쌍을 mini-batch(미니배치)로 묶음
```




In [8]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# NSMC 문장을 token ID로 바꾼 뒤 Dataset에 전달
def make_loader(processor, data, vocab_path, shuffle):
    text_tensor, _, _ = sp_tokenize(
        processor,
        data["document"].tolist(),
        vocab_path,
    )
    label_tensor = torch.tensor(data["label"].values, dtype=torch.long)
    dataset = SimpleDataset(text_tensor, label_tensor)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)


unigram_train_loader = make_loader(s, train_data, "./korean_spm.vocab", True)
unigram_validation_loader = make_loader(s, validation_data, "./korean_spm.vocab", False)
unigram_test_loader = make_loader(s, test_data, "./korean_spm.vocab", False)

sample_batch, sample_labels = next(iter(unigram_train_loader))
print("문장 batch shape:", sample_batch.shape)
print("label batch shape:", sample_labels.shape)


문장 batch shape: torch.Size([512, 80])
label batch shape: torch.Size([512])


### 6. 가장 단순한 감정 분류기


```text
nn.Module 상속 → __init__에서 Layer 선언 → forward에서 데이터 흐름 작성
```


```text
token ID → Embedding → padding을 제외한 평균 → Linear → 긍정/부정
```

- `Embedding`: 각 token ID를 256개 숫자로 표현
- `Linear`: 256개 숫자를 부정 0 / 긍정 1의 두 점수로 변환
- `mask`와 평균: 여러 token vector를 문장 하나의 vector로 만듬
- `unsqueeze(-1)`: `[batch, 길이]` mask를 `[batch, 길이, 1]`로 변경
- `clamp(min=1)`: 빈 문장이 들어와도 0으로 나누지 않게 보호


In [9]:
class SimpleSentimentModel(nn.Module):
    def __init__(self, vocab_size, pad_id):
        super().__init__()

        self.pad_id = pad_id
        embedding_dim = 256  # 먼저 정의

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=pad_id,
        )
        self.output = nn.Linear(embedding_dim, 2)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)

        mask = (token_ids != self.pad_id).unsqueeze(-1)
        token_sum = (embedded * mask).sum(dim=1)
        token_count = mask.sum(dim=1).clamp(min=1)
        sentence_vector = token_sum / token_count

        return self.output(sentence_vector)


### 7. 모델 학습과 평가


```text
train_one_epoch() : 한 epoch 동안 weight(가중치) 업데이트
evaluate()        : weight를 바꾸지 않고 loss와 accuracy 계산
```

mini-batch의 다섯 단계

```text
1. pred = model(x)              forward(예측)
2. loss = loss_fn(pred, y)      오차 계산
3. optimizer.zero_grad()        이전 gradient 지우기
4. loss.backward()              이번 gradient 계산
5. optimizer.step()             weight 업데이트
```


In [10]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()                          # 학습 모드
    total_loss = 0.0

    for x, y in loader:
        x, y = x.to(device), y.to(device)  # 데이터도 GPU/CPU로

        pred = model(x)                    # 1. forward
        loss = loss_fn(pred, y)            # 2. loss

        optimizer.zero_grad()              # 3. gradient 초기화
        loss.backward()                    # 4. backward
        optimizer.step()                   # 5. parameter 업데이트

        total_loss += loss.item()

    return total_loss / len(loader)


#  7장의 evaluate()
def evaluate(model, loader, loss_fn, device):
    model.eval()                           # 평가 모드
    total_loss, correct = 0.0, 0

    with torch.no_grad():                  # gradient 추적 끄기
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            total_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)


# 3 epoch 반복하고 validation 결과를 출력
def train_model(processor, train_loader, validation_loader, name):
    torch.manual_seed(SEED)
    model = SimpleSentimentModel(
        processor.get_piece_size(),
        processor.pad_id(),
    ).to(DEVICE)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=2e-3)
    validation_accuracy = 0.0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model, train_loader, loss_fn, optimizer, DEVICE
        )
        validation_loss, validation_accuracy = evaluate(
            model, validation_loader, loss_fn, DEVICE
        )

        print(
            f"{name} epoch {epoch}: "
            f"train_loss={train_loss:.4f}, "
            f"validation_loss={validation_loss:.4f}, "
            f"validation accuracy={validation_accuracy:.3%}"
        )

    return model, validation_accuracy


In [11]:
unigram_model, unigram_validation_accuracy = train_model(
    s,
    unigram_train_loader,
    unigram_validation_loader,
    "Unigram-8000",
)

Unigram-8000 epoch 1: train_loss=0.4865, validation_loss=0.3840, validation accuracy=83.380%
Unigram-8000 epoch 2: train_loss=0.3576, validation_loss=0.3582, validation accuracy=84.840%
Unigram-8000 epoch 3: train_loss=0.3369, validation_loss=0.3589, validation accuracy=84.660%


### 8. `model_type=bpe`와 비교

이번에는 corpus와 `vocab_size=8000`은 그대로 두고 `--model_type=bpe`만 추가합니다. 즉, 성능 차이가 생기면 핵심 변경점은 Unigram과 BPE의 token 구성 방식입니다.

In [12]:
# LMS 학습 코드에서 model_prefix와 model_type만 변경합니다.
spm.SentencePieceTrainer.Train(
    '--input={} --model_prefix=korean_spm_bpe --vocab_size={} '
    '--model_type=bpe --pad_id=3 --minloglevel=2'.format(
        temp_file, vocab_size
    )
)

bpe_s = spm.SentencePieceProcessor()
bpe_s.Load('korean_spm_bpe.model')

print("Unigram:", s.EncodeAsPieces('아버지가방에들어가신다.'))
print("BPE:", bpe_s.EncodeAsPieces('아버지가방에들어가신다.'))

Unigram: ['▁아버지', '가', '방', '에', '들어', '가', '신', '다', '.']
BPE: ['▁아버', '지가', '방', '에', '들어', '가', '신', '다', '.']


In [13]:
bpe_train_loader = make_loader(
    bpe_s, train_data, "./korean_spm_bpe.vocab", True
)
bpe_validation_loader = make_loader(
    bpe_s, validation_data, "./korean_spm_bpe.vocab", False
)

bpe_test_loader = make_loader(
    bpe_s, test_data, "./korean_spm_bpe.vocab", False
)

bpe_model, bpe_validation_accuracy = train_model(
    bpe_s,
    bpe_train_loader,
    bpe_validation_loader,
    "BPE-8000",
)

BPE-8000 epoch 1: train_loss=0.4779, validation_loss=0.3736, validation accuracy=84.120%
BPE-8000 epoch 2: train_loss=0.3566, validation_loss=0.3535, validation accuracy=85.180%
BPE-8000 epoch 3: train_loss=0.3379, validation_loss=0.3538, validation accuracy=85.220%


### 9. 비교 결과와 최종 test accuracy

두 tokenizer는 validation accuracy로 비교합니다.


In [14]:
sample_texts = validation_data["document"].iloc[:1000]
unigram_mean_tokens = np.mean([len(s.EncodeAsIds(text)) for text in sample_texts])
bpe_mean_tokens = np.mean([len(bpe_s.EncodeAsIds(text)) for text in sample_texts])

comparison = pd.DataFrame({
    "tokenizer": ["Unigram-8000", "BPE-8000"],
    "평균 token 수": [unigram_mean_tokens, bpe_mean_tokens],
    "validation accuracy": [
        unigram_validation_accuracy,
        bpe_validation_accuracy,
    ],
})
display(comparison)

# LMS evaluate() 형태대로 loss와 accuracy를 함께 계산합니다.
# 모델 선택과 비교가 끝난 뒤 주 모델을 test data에서 마지막 한 번 평가합니다.
test_loss_fn = nn.CrossEntropyLoss()
test_loss, test_accuracy = evaluate(
    unigram_model,
    unigram_test_loader,
    test_loss_fn,
    DEVICE,
)

bpe_test_loss, bpe_test_accuracy = evaluate(
    bpe_model,
    bpe_test_loader,
    test_loss_fn,
    DEVICE,
)


print(f"Unigram-8000 test loss: {test_loss:.4f}")
print(f"Unigram-8000 test accuracy: {test_accuracy:.3%}")
print(f"BPE-8000 test loss: {bpe_test_loss:.4f}")
print(f"BPE-8000 test accuracy: {bpe_test_accuracy:.3%}")

,tokenizer,평균 token 수,validation accuracy
0,Unigram-8000,17.366,0.8466
1,BPE-8000,17.032,0.8522


Unigram-8000 test loss: 0.3718
Unigram-8000 test accuracy: 84.504%
BPE-8000 test loss: 0.3715
BPE-8000 test accuracy: 84.438%


## vocab_size를 수동 변경 반복 실행 결과
---
Unigram-4000 test loss: 0.3899  
Unigram-4000 test accuracy: 83.408%  
BPE-4000 test loss: 0.3914   
BPE-4000 test accuracy: 83.443%  

---
Unigram-8000 test loss: 0.3718  
Unigram-8000 test accuracy: 84.504%  
BPE-8000 test loss: 0.3715  
BPE-8000 test accuracy: 84.438%  

---
Unigram-12000 test loss: 0.3667  
Unigram-12000 test accuracy: 84.849%  
BPE-12000 test loss: 0.3668  
BPE-12000 test accuracy: 84.820%  

---
Unigram-20000 test loss: 0.3690  
Unigram-20000 test accuracy: 84.825%  
BPE-20000 test loss: 0.3710  
BPE-20000 test accuracy: 84.674%  

---
Unigram-30000 test loss: 0.3706  
Unigram-30000 test accuracy: 84.847%  
BPE-30000 test loss: 0.3744  
BPE-30000 test accuracy: 84.763%  

---

vocab_size 12000 이후 test accuracy 변화가 정체되는 것을 확인
unigreen이 BPE에 비해 미세하게 우세한 것을 확인

## 결과 해석

1. `Unigram-8000`의 validation accuracy는 **84.66%**, `BPE-8000`은 **85.220%**였다.
2. 평균 token 수는 Unigram이 **17.366개**, BPE가 **17.032개**였다.



> NSMC train 문장을 SentencePiece에 주어 8,000개의 subword vocabulary를 학습했습니다. 문장을 token ID로 바꾸고 padding한 뒤 각 token embedding의 평균을 이용해 긍정과 부정을 분류했습니다. Unigram과 BPE를 같은 조건으로 비교했고, 주 모델의 test accuracy가 80%를 넘는지 확인했습니다.


### 코드에 사용된 개념들을 언제 어디서 배웠는가?

| 코드 내용 | LMS 노드 | 적용 방법 |
|---|---|---|
| SentencePiece 학습·사용 | 현재 프로젝트 Step 2~3 | NSMC corpus, PAD 분리, UTF-8 |
| `SimpleDataset`, `DataLoader` | 「PyTorch와 텐서 첫걸음」 6장 | 입력을 영화 리뷰 token ID로 변경 |
| `nn.Module`, `forward`, `nn.Linear` | 「PyTorch와 텐서 첫걸음」 5장 | 앞에 `Embedding`과 token 평균 추가 |
| `CrossEntropyLoss`, Optimizer | 「PyTorch와 텐서 첫걸음」 5장 | 기존 검증값을 보존해 `AdamW`, `lr=2e-3` 유지 |
| `train_one_epoch`, `evaluate` | 「PyTorch와 텐서 첫걸음」 7장 | 이미지 대신 리뷰 token ID 사용 |

#FASTTEST 적용 실습

In [15]:
# 선택 실습: True로 바꾸면 fastText Word Vector를 학습합니다.
RUN_FASTTEXT = True
FASTTEXT_REVIEW_LIMIT = len(train_data)
FASTTEXT_VECTOR_SIZE = 256

if not RUN_FASTTEXT:
    print("SKIP | fastText 실습을 실행하려면 RUN_FASTTEXT=True로 바꾸세요.")
else:
    import subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "gensim==4.4.0"
    ])
    from gensim.models import FastText

    # SentencePiece와 마찬가지로 train corpus만 사용해 data leakage를 막습니다.
    fasttext_corpus = [
        str(text).split()
        for text in train_data["document"].iloc[:FASTTEXT_REVIEW_LIMIT]
        if str(text).split()
    ]

    fasttext_model = FastText(
        sentences=fasttext_corpus,
      vector_size=FASTTEXT_VECTOR_SIZE,
      window=5,
      min_count=1,
      max_final_vocab=8000,
      workers=4,
      sg=1,
      epochs=5,
      seed=SEED,
      bucket=500_000,
    )

    sample_word = next(
        (word for word in ["영화", "재미", "배우"]
         if word in fasttext_model.wv.key_to_index),
        fasttext_model.wv.index_to_key[0],
    )
    sample_word_vector = fasttext_model.wv[sample_word]

    print("FastText vocabulary size:", len(fasttext_model.wv.key_to_index))
    print("sample word:", sample_word)
    print("word vector shape:", sample_word_vector.shape)
    print("비슷한 단어:", fasttext_model.wv.most_similar(sample_word, topn=5))

    # Vocabulary에 없는 단어도 문자 n-gram으로 Vector를 만들 수 있습니다.
    oov_word = "영화최고였당"
    print("OOV가 저장 vocabulary에 있는가?:",
          oov_word in fasttext_model.wv.key_to_index)
    print("OOV vector shape:", fasttext_model.wv[oov_word].shape)

    def fasttext_sentence_vector(sentence):
        words = str(sentence).split()
        if not words:
            return np.zeros(FASTTEXT_VECTOR_SIZE, dtype=np.float32)
        return np.mean(
            [fasttext_model.wv[word] for word in words],
            axis=0,
        )

    sample_review = "이 영화는 정말 재미있어요"
    sample_review_vector = fasttext_sentence_vector(sample_review)
    print("review:", sample_review)
    print("sentence vector shape:", sample_review_vector.shape)
    print("sentence vector 앞 5개 값:", sample_review_vector[:5])

    assert sample_word_vector.shape == (FASTTEXT_VECTOR_SIZE,)
    assert sample_review_vector.shape == (FASTTEXT_VECTOR_SIZE,)


FastText vocabulary size: 7939
sample word: 영화
word vector shape: (256,)
비슷한 단어: [('영화ㅋ', 0.8618434071540833), ('영화고', 0.8524122834205627), ('영화네', 0.8477343320846558), ('영화엔', 0.8476278781890869), ('영화네.', 0.8458028435707092)]
OOV가 저장 vocabulary에 있는가?: False
OOV vector shape: (256,)
review: 이 영화는 정말 재미있어요
sentence vector shape: (256,)
sentence vector 앞 5개 값: [ 0.23747948 -0.06191508  0.2292707   0.01340889  0.04804264]


In [16]:
# FastText vocabulary를 학습 가능한 Embedding용 ID 사전으로 변환
FT_PAD_ID = 0
FT_UNK_ID = 1
FT_DIM = fasttext_model.wv.vector_size

assert FT_DIM == FASTTEXT_VECTOR_SIZE

# 특수 토큰과 충돌하지 않는 단어만 사용
ft_words = [
    word
    for word in fasttext_model.wv.index_to_key
    if word not in {"<pad>", "<unk>"}
]

ft_word_index = {
    "<pad>": FT_PAD_ID,
    "<unk>": FT_UNK_ID,
}

for word in ft_words:
    ft_word_index[word] = len(ft_word_index)

# FastText가 학습한 Vector를 Embedding 초기값으로 복사
ft_embedding_weight = torch.zeros(
    len(ft_word_index),
    FT_DIM,
    dtype=torch.float32,
)

# 모르는 단어에는 전체 단어 Vector의 평균 사용
ft_embedding_weight[FT_UNK_ID] = torch.from_numpy(
    fasttext_model.wv.vectors.mean(axis=0).copy()
)

for word, index in ft_word_index.items():
    if index >= 2:
        ft_embedding_weight[index] = torch.from_numpy(
            fasttext_model.wv[word].copy()
        )


def encode_fasttext_ids(sentence):
    words = str(sentence).split()[:MAX_LENGTH]

    token_ids = [
        ft_word_index.get(word, FT_UNK_ID)
        for word in words
    ]

    # 빈 문장은 <unk> 하나로 처리
    if not token_ids:
        token_ids = [FT_UNK_ID]

    return torch.tensor(token_ids, dtype=torch.long)


def make_fasttext_finetune_loader(data, shuffle):
    sequences = [
        encode_fasttext_ids(sentence)
        for sentence in data["document"]
    ]

    text_tensor = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=FT_PAD_ID,
    )

    label_tensor = torch.tensor(
        data["label"].to_numpy(),
        dtype=torch.long,
    )

    dataset = SimpleDataset(text_tensor, label_tensor)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )

In [17]:
class FastTextFineTuneModel(nn.Module):
    def __init__(self, embedding_weight, pad_id):
        super().__init__()

        self.pad_id = pad_id

        self.embedding = nn.Embedding.from_pretrained(
            embedding_weight,
            freeze=False,          # 핵심: FastText Vector도 함께 학습
            padding_idx=pad_id,
        )

        self.output = nn.Linear(
            embedding_weight.shape[1],
            2,
        )

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)

        # Padding을 제외하고 단어 Vector 평균 계산
        mask = (token_ids != self.pad_id).unsqueeze(-1)
        token_sum = (embedded * mask).sum(dim=1)
        token_count = mask.sum(dim=1).clamp(min=1)
        sentence_vector = token_sum / token_count

        return self.output(sentence_vector)


fasttext_finetune_train_loader = make_fasttext_finetune_loader(
    train_data, True
)
fasttext_finetune_validation_loader = make_fasttext_finetune_loader(
    validation_data, False
)
fasttext_finetune_test_loader = make_fasttext_finetune_loader(
    test_data, False
)

torch.manual_seed(SEED)

fasttext_finetuned_model = FastTextFineTuneModel(
    ft_embedding_weight,
    FT_PAD_ID,
).to(DEVICE)

fasttext_finetuned_loss_fn = nn.CrossEntropyLoss()
fasttext_finetuned_optimizer = optim.AdamW(
    fasttext_finetuned_model.parameters(),
    lr=2e-3,
)

print(
    "학습 가능한 parameter 수:",
    sum(
        parameter.numel()
        for parameter in fasttext_finetuned_model.parameters()
        if parameter.requires_grad
    ),
)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(
        fasttext_finetuned_model,
        fasttext_finetune_train_loader,
        fasttext_finetuned_loss_fn,
        fasttext_finetuned_optimizer,
        DEVICE,
    )

    validation_loss, fasttext_finetuned_validation_accuracy = evaluate(
        fasttext_finetuned_model,
        fasttext_finetune_validation_loader,
        fasttext_finetuned_loss_fn,
        DEVICE,
    )

    print(
        f"FastText-finetuned epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"validation_loss={validation_loss:.4f}, "
        f"validation accuracy="
        f"{fasttext_finetuned_validation_accuracy:.3%}"
    )

fasttext_finetuned_test_loss, fasttext_finetuned_test_accuracy = evaluate(
    fasttext_finetuned_model,
    fasttext_finetune_test_loader,
    fasttext_finetuned_loss_fn,
    DEVICE,
)

print(
    f"FastText-finetuned test loss: "
    f"{fasttext_finetuned_test_loss:.4f}"
)
print(
    f"FastText-finetuned test accuracy: "
    f"{fasttext_finetuned_test_accuracy:.3%}"
)

학습 가능한 parameter 수: 2033410
FastText-finetuned epoch 1: train_loss=0.5070, validation_loss=0.4680, validation accuracy=76.000%
FastText-finetuned epoch 2: train_loss=0.4459, validation_loss=0.4709, validation accuracy=75.020%
FastText-finetuned epoch 3: train_loss=0.4360, validation_loss=0.4720, validation accuracy=75.260%
FastText-finetuned test loss: 0.4747
FastText-finetuned test accuracy: 75.828%


In [18]:
model_test_results = {
    "Unigram-8000": test_accuracy,
    "BPE-8000": bpe_test_accuracy,
    "FastText-finetuned": fasttext_finetuned_test_accuracy,
}

print("# 세 모델 Test Accuracy 비교\n")

for model_name, accuracy in model_test_results.items():
    print(f"{model_name:<20}: {accuracy:.3%}")

best_model = max(
    model_test_results,
    key=model_test_results.get,
)

print("-" * 35)
print("가장 높은 모델:", best_model)
print(
    "가장 높은 정확도:",
    f"{model_test_results[best_model]:.3%}",
)

# 세 모델 Test Accuracy 비교

Unigram-8000        : 84.504%
BPE-8000            : 84.438%
FastText-finetuned  : 75.828%
-----------------------------------
가장 높은 모델: Unigram-8000
가장 높은 정확도: 84.504%


처음에는 리뷰에 포함된 FastText Linear 분류기만 학습했습니다.
고정된 FastText 문장 평균 Vector
→ Linear(256, 2)
이 방식에서는 학습 가능한 Parameter(매개변수)가 514개뿐이었으며, FastText 자체도 감정 Label(정답)을 보지 않고 단어의 문맥만 학습했습니다.
그 결과 Test accuracy는 58.851%에 머물렀습니다.
4. FastText 개선 방법
FastText 단어 벡터를 nn.Embedding.from_pretrained()의 초기값으로 사용하고 다음 옵션을 적용했습니다.
freeze=False
이를 통해 감정 분석을 학습하면서 FastText Embedding도 함께 Fine-tuning(미세조정)되도록 개선했습니다.
또한 다음 사항을 적용했습니다.
- <pad>=0, <unk>=1 분리
- 문장을 단어 ID 배열로 변환
- pad_sequence()로 Padding
- Padding을 제외하고 단어 벡터 평균 계산
- SentencePiece와 동일한 Embedding → 평균 → Linear 구조 사용
개선 후 FastText 정확도는 75.832%로 상승했습니다.
58.851% → 75.832%
개선 폭: +16.981%p
최종 결과
모델	Test accuracy
Unigram-8000	84.471%
BPE-8000	84.634%
FastText-finetuned	75.832%


BPE는 Unigram보다 0.163%p 높았고, FastText-finetuned보다 8.802%p 높았습니다.
결과 해석
FastText 단어 벡터를 고정하여 사용했을 때는 감정 정보가 충분히 학습되지 않아 성능이 낮았다. FastText Embedding을 감정 분석 모델과 함께 Fine-tuning하자 정확도가 16.981%p 향상되었다. 하지만 이 실험에서는 한국어의 조사, 어미, 띄어쓰기 변형을 Subword(부분 단어)로 처리하는 SentencePiece가 더 높은 성능을 보였으며, BPE-8000이 84.634%로 가장 좋은 결과를 기록했다.

#KoNLPy_MECAB 실습

In [19]:
#@title 1. KoNLPy Okt 설치 및 확인 { display-mode: "form" }

import importlib
import importlib.util
import shutil
import subprocess
import sys

# KoNLPy는 Java가 필요합니다.
if shutil.which("java") is None:
    subprocess.check_call(["apt-get", "update", "-qq"])
    subprocess.check_call([
        "apt-get", "install", "-y", "-qq",
        "default-jre-headless",
    ])

if importlib.util.find_spec("konlpy") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "konlpy==0.6.0",
    ])
    importlib.invalidate_caches()

from konlpy.tag import Okt

# 하나의 Okt 객체를 계속 재사용합니다.
okt = Okt(max_heap_size=2048)

sample_review = "이 영화는 정말 재미있었어요! 배우들도 좋았습니다."
sample_morphs = okt.morphs(
    sample_review,
    norm=True,     # 인터넷 표현 등을 정규화
    stem=True,     # 재미있었어요 → 재미있다
)

print("원문:", sample_review)
print("Okt 형태소:", sample_morphs)

원문: 이 영화는 정말 재미있었어요! 배우들도 좋았습니다.
Okt 형태소: ['이', '영화', '는', '정말', '재미있다', '!', '배우', '들', '도', '좋다', '.']


In [20]:
#@title 2. NSMC 전체 리뷰를 Okt 형태소로 변환 { display-mode: "form" }

import time
from tqdm.auto import tqdm


def tokenize_with_okt(texts, description):
    tokenized_sentences = []

    for text in tqdm(
        texts,
        total=len(texts),
        desc=description,
    ):
        tokens = okt.morphs(
            str(text),
            norm=True,
            stem=True,
        )
        tokenized_sentences.append(tokens)

    return tokenized_sentences


# Vocabulary는 아래 train 결과만 사용해서 만듭니다.
okt_started = time.perf_counter()

okt_train_tokens = tokenize_with_okt(
    train_data["document"],
    "Okt train",
)

okt_validation_tokens = tokenize_with_okt(
    validation_data["document"],
    "Okt validation",
)

okt_test_tokens = tokenize_with_okt(
    test_data["document"],
    "Okt test",
)

okt_tokenization_minutes = (
    time.perf_counter() - okt_started
) / 60

print(
    "Okt 전체 형태소 분석 시간:",
    f"{okt_tokenization_minutes:.1f}분",
)
print("첫 번째 원문:", train_data["document"].iloc[0])
print("첫 번째 형태소:", okt_train_tokens[0])

Okt train:   0%|          | 0/141182 [00:00<?, ?it/s]

Okt validation:   0%|          | 0/5000 [00:00<?, ?it/s]

Okt test:   0%|          | 0/48361 [00:00<?, ?it/s]

Okt 전체 형태소 분석 시간: 7.6분
첫 번째 원문: 아 더빙.. 진짜 짜증나네요 목소리
첫 번째 형태소: ['아', '더빙', '..', '진짜', '짜증나다', '목소리']


In [21]:
#@title 3. Okt Vocabulary와 DataLoader 생성 { display-mode: "form" }

from collections import Counter

KONLPY_VOCAB_SIZE = 8000
OKT_PAD_ID = 0
OKT_UNK_ID = 1

# 반드시 train 형태소만 세어 Vocabulary를 만듭니다.
okt_counter = Counter(
    token
    for sentence in okt_train_tokens
    for token in sentence
)

# 출현 횟수가 같으면 가나다순으로 정렬해 결과를 재현합니다.
ordered_tokens = sorted(
    okt_counter.items(),
    key=lambda item: (-item[1], item[0]),
)

okt_word_index = {
    "<pad>": OKT_PAD_ID,
    "<unk>": OKT_UNK_ID,
}

for token, count in ordered_tokens:
    if token not in okt_word_index:
        okt_word_index[token] = len(okt_word_index)

    if len(okt_word_index) >= KONLPY_VOCAB_SIZE:
        break

okt_index_word = {
    index: token
    for token, index in okt_word_index.items()
}


def encode_okt_tokens(tokens):
    token_ids = [
        okt_word_index.get(token, OKT_UNK_ID)
        for token in tokens[:MAX_LENGTH]
    ]

    if not token_ids:
        token_ids = [OKT_UNK_ID]

    return torch.tensor(
        token_ids,
        dtype=torch.long,
    )


def make_okt_loader(tokenized_sentences, data, shuffle):
    sequences = [
        encode_okt_tokens(tokens)
        for tokens in tokenized_sentences
    ]

    text_tensor = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=OKT_PAD_ID,
    )

    label_tensor = torch.tensor(
        data["label"].to_numpy(),
        dtype=torch.long,
    )

    dataset = SimpleDataset(
        text_tensor,
        label_tensor,
    )

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
    )


okt_train_loader = make_okt_loader(
    okt_train_tokens,
    train_data,
    True,
)

okt_validation_loader = make_okt_loader(
    okt_validation_tokens,
    validation_data,
    False,
)

okt_test_loader = make_okt_loader(
    okt_test_tokens,
    test_data,
    False,
)

okt_sample_batch, okt_sample_labels = next(
    iter(okt_train_loader)
)

print("Okt Vocabulary 크기:", len(okt_word_index))
print("문장 Batch shape:", okt_sample_batch.shape)
print("Label Batch shape:", okt_sample_labels.shape)
print("PAD ID:", OKT_PAD_ID)
print("UNK ID:", OKT_UNK_ID)

assert len(okt_word_index) <= KONLPY_VOCAB_SIZE
assert OKT_PAD_ID != OKT_UNK_ID
assert okt_sample_batch.dtype == torch.long
assert okt_sample_batch.shape[1] <= MAX_LENGTH

Okt Vocabulary 크기: 8000
문장 Batch shape: torch.Size([512, 80])
Label Batch shape: torch.Size([512])
PAD ID: 0
UNK ID: 1


In [22]:
#@title 4. KoNLPy Okt 감정 분류 모델 학습 { display-mode: "form" }

torch.manual_seed(SEED)

# SentencePiece에 사용한 모델을 그대로 재사용합니다.
okt_model = SimpleSentimentModel(
    vocab_size=len(okt_word_index),
    pad_id=OKT_PAD_ID,
).to(DEVICE)

# 두 모델의 Embedding 차원이 같은지 확인합니다.
print(
    "SentencePiece Embedding 차원:",
    unigram_model.embedding.embedding_dim,
)
print(
    "Okt Embedding 차원:",
    okt_model.embedding.embedding_dim,
)

assert (
    okt_model.embedding.embedding_dim
    == unigram_model.embedding.embedding_dim
)

okt_loss_fn = nn.CrossEntropyLoss()
okt_optimizer = optim.AdamW(
    okt_model.parameters(),
    lr=2e-3,
)

okt_validation_accuracy = 0.0

for epoch in range(1, EPOCHS + 1):
    okt_train_loss = train_one_epoch(
        okt_model,
        okt_train_loader,
        okt_loss_fn,
        okt_optimizer,
        DEVICE,
    )

    (
        okt_validation_loss,
        okt_validation_accuracy,
    ) = evaluate(
        okt_model,
        okt_validation_loader,
        okt_loss_fn,
        DEVICE,
    )

    print(
        f"KoNLPy-Okt-8000 epoch {epoch}: "
        f"train_loss={okt_train_loss:.4f}, "
        f"validation_loss={okt_validation_loss:.4f}, "
        f"validation accuracy={okt_validation_accuracy:.3%}"
    )

# 설정을 모두 결정한 뒤 test data는 마지막에 평가합니다.
okt_test_loss, okt_test_accuracy = evaluate(
    okt_model,
    okt_test_loader,
    okt_loss_fn,
    DEVICE,
)

print("=" * 45)
print(f"KoNLPy-Okt-8000 test loss: {okt_test_loss:.4f}")
print(f"KoNLPy-Okt-8000 test accuracy: {okt_test_accuracy:.3%}")

SentencePiece Embedding 차원: 256
Okt Embedding 차원: 256
KoNLPy-Okt-8000 epoch 1: train_loss=0.4623, validation_loss=0.3836, validation accuracy=83.900%
KoNLPy-Okt-8000 epoch 2: train_loss=0.3655, validation_loss=0.3668, validation accuracy=84.980%
KoNLPy-Okt-8000 epoch 3: train_loss=0.3476, validation_loss=0.3629, validation accuracy=84.880%
KoNLPy-Okt-8000 test loss: 0.3809
KoNLPy-Okt-8000 test accuracy: 84.161%


In [23]:
#@title 5. 네 모델 Test Accuracy 비교

model_test_results = {
    "Unigram-8000": test_accuracy,
    "BPE-8000": bpe_test_accuracy,
    "KoNLPy-Okt-8000": okt_test_accuracy,
}

# FastText 셀을 실행한 경우에만 결과에 추가합니다.
if "fasttext_finetuned_test_accuracy" in globals():
    model_test_results["FastText-finetuned"] = (
        fasttext_finetuned_test_accuracy
    )

print("# 모델별 Test Accuracy 비교\n")

for model_name, accuracy in model_test_results.items():
    print(f"{model_name:<22}: {accuracy:.3%}")

best_model = max(
    model_test_results,
    key=model_test_results.get,
)

print("-" * 40)
print("가장 높은 모델:", best_model)
print(
    "가장 높은 정확도:",
    f"{model_test_results[best_model]:.3%}",
)

# 모델별 Test Accuracy 비교

Unigram-8000          : 84.504%
BPE-8000              : 84.438%
KoNLPy-Okt-8000       : 84.161%
FastText-finetuned    : 75.828%
----------------------------------------
가장 높은 모델: Unigram-8000
가장 높은 정확도: 84.504%
